In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sympy.polys.polyconfig import query

In [15]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
    def forward(self, x):
        return self.block(x)


In [16]:
class ProtoNet(nn.Module):
    def __init__(self, in_channels=1, hidden_dim=64):
        super().__init__()

        self.encoder = nn.Sequential(
            ConvBlock(in_channels, hidden_dim),
            ConvBlock(hidden_dim, hidden_dim),
            ConvBlock(hidden_dim, hidden_dim),
            ConvBlock(hidden_dim, hidden_dim),
        )

    def forward(self, x):
        x = self.encoder(x)
        return x.view(x.size(0), -1)

In [17]:
import random

class EpisodeSampler:
    def __init__(self, data, labels, n_way, k_shot, q_query):
        self.data = data
        self.labels = labels
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_query = q_query
        self.classes = list(set(labels))

        self.class_to_indices = {}
        for idx, label in enumerate(self.labels):
            self.class_to_indices.setdefault(label, []).append(idx)


    def sample_episode(self):
        selected_classes = random.sample(self.classes, self.n_way)

        support_x, support_y = [], []
        query_x, query_y = [], []

        for i, cls in enumerate(selected_classes):
            indices = random.sample(
                self.class_to_indices[cls],
                self.k_shot + self.q_query
            )

            support_idx = indices[:self.k_shot]
            query_idx = indices[self.k_shot:]

            support_x += [self.data[j] for j in support_idx]
            query_x += [self.data[j] for j in query_idx]

            support_y += [i] * self.k_shot
            query_y += [i] * self.q_query

        return (
            torch.stack(support_x),
            torch.tensor(support_y),
            torch.stack(query_x),
            torch.tensor(query_y),
        )


In [18]:
def compute_prototypes(embeddings, labels, n_way):
    prototypes = []
    for c in range(n_way):
        class_embeddings = embeddings[labels == c]
        prototypes.append(class_embeddings.mean(dim=0))
    return torch.stack(prototypes)
def euclidean_distance_squared(x, y):
    n = x.size(0)
    m = y.size(0)

    x = x.unsqueeze(1).expand(n, m, -1)
    y = y.unsqueeze(0).expand(n, m, -1)

    return ((x - y) ** 2).sum(2)

In [19]:
def train(model, sampler, optimizer, episodes=10000):
    model.train()

    for episode in range(episodes):
        support_x, support_y, query_x, query_y = sampler.sample_episode()

        support_emb = model(support_x)
        query_emb = model(query_x)

        prototypes = compute_prototypes(support_emb, support_y, sampler.n_way)

        distances = euclidean_distance_squared(query_emb, prototypes)

        loss = F.cross_entropy(-distances, query_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if episode % 100 == 0:
            pred = torch.argmin(distances, dim=1)
            acc = (pred == query_y).float().mean().item()

            print(f"Episode {episode} | Loss: {loss.item():.4f} | Accuracy: {acc:.4f}")

In [28]:
import numpy as np

def evaluate(model, sampler, episodes=600):
    model.eval()

    accuracies = []

    with torch.no_grad():
        for _ in range(episodes):
            support_x, support_y, query_x, query_y = sampler.sample_episode()

            support_emb = model(support_x)
            query_emb = model(query_x)

            prototypes = compute_prototypes(support_emb, support_y, sampler.n_way)
            distances = euclidean_distance_squared(query_emb, prototypes)

            pred = torch.argmin(distances, dim=1)
            acc = (pred == query_y).float().mean().item()
            accuracies.append(acc)

    mean = np.mean(accuracies)
    std = np.std(accuracies)
    ci95 = 1.96 * std / np.sqrt(episodes)

    print(f"Accuracy: {mean:.4f} ± {ci95:.4f}")

    return mean, ci95, accuracies

In [ ]:
from io import BytesIO
import torch
from torchvision import transforms, datasets
from datasets import load_dataset
from PIL import Image
import torch.nn.functional as F
import numpy as np

transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((28,28)),
    transforms.ToTensor()
])

omniglot_train = datasets.Omniglot(
    root="./data",
    background=True,
    download=True,
    transform=transform
)

ds = load_dataset("imagefolder", split='train', data_dir="./geez_data/OCR_dataset/train")

geez_imgs, geez_labels = [], []

for i in range(ds.num_rows):

    img = ds[i]["image"]

    img = transform(img)
    geez_imgs.append(img)
    geez_labels.append(int(ds[i]["label"]))



geez_test = list(zip(geez_imgs, geez_labels))

acc_l = []

for k_shot in [1, 5]:
    print(f"\n=== {k_shot}-shot Experiment ===")

    model = ProtoNet()

    train_sampler = EpisodeSampler([x for x, _ in omniglot_train], n_way=5, k_shot=k_shot, q_query=15, labels=[x for _, x in omniglot_train])
    test_sampler  = EpisodeSampler([x for x, _ in geez_test], n_way=5, k_shot=k_shot, q_query=15, labels=[x for _, x in geez_test])

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Train using existing train() function
    train(model, train_sampler, optimizer, episodes=600)

    # Evaluate on Ge'ez using evaluate() function
    _, __, acc = evaluate(model, test_sampler, episodes=600)
    acc_l.append(acc)

import numpy as np

def compute_accuracy_over_time(correct_list):
    accuracies = []

    for i in range(1, len(correct_list) + 1):
        window_slice = correct_list[:i]
        acc = np.mean(window_slice)
        accuracies.append(acc)

    return accuracies

import matplotlib.pyplot as plt

episodes = list(range(len(acc_l)))  # e.g., 0 to 599
acc_list_1shot = compute_accuracy_over_time(acc[0])  # your stored accuracy at each evaluation point
acc_list_5shot = compute_accuracy_over_time(acc[1])  # same for 5-shot

plt.figure(figsize=(8,5))
plt.plot(episodes, acc_list_1shot, label="1-shot", color="blue")
plt.plot(episodes, acc_list_5shot, label="5-shot", color="orange")
plt.xlabel("Episode")
plt.ylabel("Query Set Accuracy")
plt.title("ProtoNet: Accuracy over Episodes")
plt.legend()
plt.grid(True)
plt.show()

In [22]:
class MAML(nn.Module):
    def __init__(self, in_channels=1, hidden_dim=64, n_way=5):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, hidden_dim, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(hidden_dim)

        self.conv2 = nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(hidden_dim)

        self.conv3 = nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1)
        self.bn3   = nn.BatchNorm2d(hidden_dim)

        self.conv4 = nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1)
        self.bn4   = nn.BatchNorm2d(hidden_dim)

        self.classifier = nn.Linear(hidden_dim, n_way)


    def forward(self, x, params=None):
        if params is None:
            params = OrderedDict(self.named_parameters())

        # Block 1
        x = F.conv2d(x, params['conv1.weight'], params['conv1.bias'], padding=1)
        x = F.batch_norm(
            x, None, None,
            weight=params['bn1.weight'],
            bias=params['bn1.bias'],
            training=True
        )
        x = F.relu(x)
        x = F.max_pool2d(x, 2)

        # Block 2
        x = F.conv2d(x, params['conv2.weight'], params['conv2.bias'], padding=1)
        x = F.batch_norm(
            x, None, None,
            weight=params['bn2.weight'],
            bias=params['bn2.bias'],
            training=True
        )
        x = F.relu(x)
        x = F.max_pool2d(x, 2)

        # Block 3
        x = F.conv2d(x, params['conv3.weight'], params['conv3.bias'], padding=1)
        x = F.batch_norm(
            x, None, None,
            weight=params['bn3.weight'],
            bias=params['bn3.bias'],
            training=True
        )
        x = F.relu(x)
        x = F.max_pool2d(x, 2)

        # Block 4
        x = F.conv2d(x, params['conv4.weight'], params['conv4.bias'], padding=1)
        x = F.batch_norm(
            x, None, None,
            weight=params['bn4.weight'],
            bias=params['bn4.bias'],
            training=True
        )
        x = F.relu(x)
        x = F.max_pool2d(x, 2)

        x = x.view(x.size(0), -1)

        logits = F.linear(
            x,
            params['classifier.weight'],
            params['classifier.bias']
        )

        return logits

In [23]:
def inner_loop(model, support_x, support_y, inner_steps=5, inner_lr=0.01):
    params = OrderedDict(model.named_parameters())

    for step in range(inner_steps):

        forward_pass = model(support_x, params)

        loss = F.cross_entropy(forward_pass, support_y)

        grads = torch.autograd.grad(
            loss,
            params.values(),
            create_graph=True
        )


        params = OrderedDict(
            (name, param - inner_lr * grad)
            for ((name, param), grad) in zip(params.items(), grads)
        )



    return params


In [24]:
def train_maml(model, sampler, optimizer, episodes=1000, inner_steps=5, inner_lr=0.01):
    model.train()

    for episode in range(episodes):
        support_x, support_y, query_x, query_y = sampler.sample_episode()

        adapted_params = inner_loop(model, support_x, support_y, inner_steps, inner_lr)

        query_logits = model(query_x, adapted_params)
        query_logits = query_logits.view(query_logits.size(0), -1)
        loss_query = F.cross_entropy(query_logits, query_y)

        optimizer.zero_grad()
        loss_query.backward()
        optimizer.step()

        if episode % 100 == 0:
            acc = (query_logits.argmax(dim=1) == query_y).float().mean().item()
            print(f"Episode {episode:04d} | Query Loss: {loss_query.item():.4f} | Query Acc: {acc:.4f}")


In [25]:
def evaluate_maml(model, sampler, episodes=600, inner_steps=5, inner_lr=0.01):
    model.eval()
    accuracies = []

    for _ in range(episodes):
        support_x, support_y, query_x, query_y = sampler.sample_episode()

        adapted_params = inner_loop(model, support_x, support_y, inner_steps, inner_lr)

        logits = model(query_x, adapted_params)
        pred = torch.argmax(logits, dim=1)
        acc = (pred == query_y).float().mean().item()
        accuracies.append(acc)

    mean = np.mean(accuracies)
    std = np.std(accuracies)
    ci95 = 1.96 * std / np.sqrt(episodes)

    print(f"Accuracy: {mean:.4f} ± {ci95:.4f}")
    return mean, ci95, accuracies

In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets
from datasets import load_dataset
from collections import OrderedDict
import numpy as np
import random


transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((28, 28)),
    transforms.ToTensor()
])

omniglot_train = datasets.Omniglot(
    root="./data",
    background=True,
    download=True,
    transform=transform
)

train_data = [x for x, _ in omniglot_train]
train_labels = [y for _, y in omniglot_train]

ds = load_dataset("imagefolder", split="train", data_dir="./geez_data/OCR_dataset/train")

geez_imgs, geez_labels = [], []

for i in range(ds.num_rows):
    img = transform(ds[i]["image"])
    geez_imgs.append(img)
    geez_labels.append(int(ds[i]["label"]))

acc_l = []

for k_shot in [1, 5]:
    print(f"\n=== {k_shot}-shot MAML ===")

    model = MAML(n_way=5)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    train_sampler = EpisodeSampler(train_data, train_labels, 5, k_shot, 15)
    test_sampler  = EpisodeSampler(geez_imgs, geez_labels, 5, k_shot, 15)

    train_maml(model, train_sampler, optimizer, episodes=600)
    _, __, acc = evaluate_maml(model, test_sampler, episodes=600)
    acc_l.append(acc)

import numpy as np

def compute_accuracy_over_time(correct_list):
    accuracies = []

    for i in range(1, len(correct_list) + 1):
        window_slice = correct_list[:i]
        acc = np.mean(window_slice)
        accuracies.append(acc)

    return accuracies

import matplotlib.pyplot as plt

episodes = list(range(len(acc_l)))  # e.g., 0 to 599
acc_list_1shot = compute_accuracy_over_time(acc[0])  # your stored accuracy at each evaluation point
acc_list_5shot = compute_accuracy_over_time(acc[1])  # same for 5-shot

plt.figure(figsize=(8,5))
plt.plot(episodes, acc_list_1shot, label="1-shot", color="blue")
plt.plot(episodes, acc_list_5shot, label="5-shot", color="orange")
plt.xlabel("Episode")
plt.ylabel("Query Set Accuracy")
plt.title("ProtoNet: Accuracy over Episodes")
plt.legend()
plt.grid(True)
plt.show()


=== 1-shot MAML ===
Episode 0000 | Query Loss: 1.1109 | Query Acc: 0.7067
Episode 0100 | Query Loss: 1.1391 | Query Acc: 0.6000
Episode 0200 | Query Loss: 0.4000 | Query Acc: 0.9333
Episode 0300 | Query Loss: 0.6051 | Query Acc: 0.8267
Episode 0400 | Query Loss: 0.4559 | Query Acc: 0.8667
Episode 0500 | Query Loss: 0.5651 | Query Acc: 0.8000
Accuracy: 0.3586 ± 0.0069

=== 5-shot MAML ===
Episode 0000 | Query Loss: 1.1201 | Query Acc: 0.7467
Episode 0100 | Query Loss: 0.4318 | Query Acc: 0.8933
Episode 0200 | Query Loss: 0.3174 | Query Acc: 0.9067
Episode 0300 | Query Loss: 0.2852 | Query Acc: 0.8933


KeyboardInterrupt: 